# SupportFlow AI: RNN, LSTM і Transformer

Мета: з маленького корпусу пар `question -> answer` підготувати токени, навчити просту RNN і LSTM для генерації коротких відповідей, а потім порівняти їх із переднавченою Transformer-моделлю.


## Залежності

Якщо імпорти нижче не проходять, встановіть пакети в окремій клітинці:

```python
%pip install torch pandas transformers
```


## Залежності та фіксація випадковості

`torch.manual_seed(42)` робить ініціалізацію ваг повторюваною. Це не гарантує ідентичні результати на кожному пристрої, але прибирає випадковість, яка заважає порівнювати RNN і LSTM.

У notebook використовується CPU/GPU автоматично:

- `cuda` - якщо доступна відеокарта NVIDIA;
- `mps` - якщо доступний Apple Silicon backend;
- `cpu` - універсальний варіант.


In [1]:
from pathlib import Path
import re
import string
import time
from collections import Counter

import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F


SEED = 42
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)


PyTorch: 2.12.0
Device: mps


## 1. Корпус запитів і відповідей

Корпус невеликий спеціально: так легше побачити весь пайплайн без довгого тренування.

Модель не розуміє текст як слова напряму. Спочатку кожне слово стане числовим індексом, а вже ці індекси будуть подаватися в нейромережу.


In [2]:
qa_pairs = [
    ("How can I reset my password?", "You can reset your password in account settings."),
    ("I forgot my password", "Use the password reset link on the login page."),
    ("How do I change my email?", "You can update your email in profile settings."),
    ("Where is my order?", "You can track your order from the orders page."),
    ("My order is delayed", "Your order may take a bit longer than expected."),
    ("Can I cancel my order?", "You can cancel the order before it is shipped."),
    ("How do I request a refund?", "Open the order details and choose request refund."),
    ("The product arrived damaged", "Please send a photo and we will help with replacement."),
    ("How can I contact support?", "You can contact support through chat or email."),
    ("I cannot log into my account", "Check your email and password or reset your password."),
    ("How do I update my address?", "You can edit your shipping address in account settings."),
    ("Do you ship internationally?", "International shipping is available for selected countries."),
    ("How long does delivery take?", "Delivery usually takes three to five business days."),
    ("Can I change my payment method?", "You can change payment method before confirming the order."),
    ("I was charged twice", "Please contact support and we will check the payment."),
    ("Where can I find my invoice?", "Your invoice is available in the billing section."),
    ("How do I delete my account?", "You can request account deletion in privacy settings."),
    ("Can I return an item?", "You can return eligible items within thirty days."),
    ("The discount code does not work", "Check the code terms or contact support for help."),
    ("How do I track a refund?", "Refund status is shown in the order details."),
    ("I received the wrong item", "Please contact support and include your order number."),
    ("Can I change the delivery date?", "You can change delivery date if the order is not shipped."),
    ("How do I subscribe to updates?", "Enable notifications in your profile preferences."),
    ("How do I unsubscribe from emails?", "Use the unsubscribe link or change email preferences."),
    ("Is my personal data secure?", "Your personal data is protected according to our privacy policy."),
]

pd.DataFrame(qa_pairs, columns=["question", "answer"]).head(10)


,question,answer
0,How can I reset my password?,You can reset your password in account settings.
1,I forgot my password,Use the password reset link on the login page.
2,How do I change my email?,You can update your email in profile settings.
3,Where is my order?,You can track your order from the orders page.
4,My order is delayed,Your order may take a bit longer than expected.
5,Can I cancel my order?,You can cancel the order before it is shipped.
6,How do I request a refund?,Open the order details and choose request refund.
7,The product arrived damaged,Please send a photo and we will help with repl...
8,How can I contact support?,You can contact support through chat or email.
9,I cannot log into my account,Check your email and password or reset your pa...


<sos> You can reset your password in account settings.
You can reset your password in account settings. <eos>

## 2. Нормалізація, токенізація і словник

Нормалізація зменшує кількість різних токенів:

- нижній регістр: `Password` і `password` стають одним словом;
- видалення пунктуації: `password?` і `password` теж стають одним словом;
- видалення чисел: корисно для узагальнення, якщо ID замовлень або суми не мають бути окремими словами.

Спеціальні токени:

- `<pad>` - заповнення коротких речень до однакової довжини;
- `<sos>` - початок відповіді для decoder;
- `<eos>` - кінець відповіді;
- `<unk>` - слово, якого немає у словнику.


In [3]:
SPECIAL_TOKENS = ["<pad>", "<sos>", "<eos>", "<unk>"]
PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN = SPECIAL_TOKENS


def normalize_text(text):
    text = text.lower()
    text = re.sub(r"\d+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    return normalize_text(text).split()


question_tokens = [tokenize(question) for question, _ in qa_pairs]
answer_tokens = [tokenize(answer) for _, answer in qa_pairs]

counter = Counter()
for tokens in question_tokens + answer_tokens:
    counter.update(tokens)

idx_to_token = SPECIAL_TOKENS + sorted(counter)
token_to_idx = {token: idx for idx, token in enumerate(idx_to_token)}

PAD_IDX = token_to_idx[PAD_TOKEN]
SOS_IDX = token_to_idx[SOS_TOKEN]
EOS_IDX = token_to_idx[EOS_TOKEN]
UNK_IDX = token_to_idx[UNK_TOKEN]
VOCAB_SIZE = len(idx_to_token)

MAX_QUESTION_LEN = max(len(tokens) for tokens in question_tokens)
MAX_ANSWER_LEN = max(len(tokens) for tokens in answer_tokens) + 1  # + <eos>

print("Vocabulary size:", VOCAB_SIZE)
print("Max question length:", MAX_QUESTION_LEN)
print("Max answer length with <eos>:", MAX_ANSWER_LEN)
print("First 20 tokens:", idx_to_token[:20])


Vocabulary size: 138
Max question length: 6
Max answer length with <eos>: 12
First 20 tokens: ['<pad>', '<sos>', '<eos>', '<unk>', 'a', 'according', 'account', 'address', 'an', 'and', 'arrived', 'available', 'before', 'billing', 'bit', 'business', 'can', 'cancel', 'cannot', 'change']


## 3. Кодування послідовностей

Для encoder використовуємо токени запиту. Для decoder під час навчання застосовується `teacher forcing`: на вхід decoder подається правильна попередня частина відповіді, а модель вчиться передбачати наступний токен.

Приклад для відповіді `you can reset password`:

- decoder input: `<sos> you can reset password`
- target: `you can reset password <eos>`

`CrossEntropyLoss` порівнює logits моделі з `target` для кожної позиції.


In [4]:
def encode_tokens(tokens, max_len, add_eos=False, add_sos=False):
    ids = []
    if add_sos:
        ids.append(SOS_IDX)
    ids.extend(token_to_idx.get(token, UNK_IDX) for token in tokens)
    if add_eos:
        ids.append(EOS_IDX)
    ids = ids[:max_len]
    ids += [PAD_IDX] * (max_len - len(ids))
    return ids


src_ids = torch.tensor(
    [encode_tokens(tokens, MAX_QUESTION_LEN) for tokens in question_tokens],
    dtype=torch.long,
    device=DEVICE,
)
decoder_input_ids = torch.tensor(
    [encode_tokens(tokens, MAX_ANSWER_LEN, add_sos=True) for tokens in answer_tokens],
    dtype=torch.long,
    device=DEVICE,
)
target_ids = torch.tensor(
    [encode_tokens(tokens, MAX_ANSWER_LEN, add_eos=True) for tokens in answer_tokens],
    dtype=torch.long,
    device=DEVICE,
)

print("src_ids:", tuple(src_ids.shape))
print("decoder_input_ids:", tuple(decoder_input_ids.shape))
print("target_ids:", tuple(target_ids.shape))


src_ids: (25, 6)
decoder_input_ids: (25, 12)
target_ids: (25, 12)


In [5]:
sample_i = 0

sample_rows = []
for role, ids in [
    ("question_tokens", src_ids[sample_i].cpu().tolist()),
    ("decoder_input", decoder_input_ids[sample_i].cpu().tolist()),
    ("target", target_ids[sample_i].cpu().tolist()),
]:
    for position, token_id in enumerate(ids):
        sample_rows.append(
            {
                "sequence": role,
                "position": position,
                "token_id": token_id,
                "token": idx_to_token[token_id],
            }
        )

pd.DataFrame(sample_rows)


,sequence,position,token_id,token
0,question_tokens,0,52,how
1,question_tokens,1,16,can
2,question_tokens,2,53,i
3,question_tokens,3,98,reset
4,question_tokens,4,72,my
5,question_tokens,5,83,password
6,decoder_input,0,1,<sos>
7,decoder_input,1,136,you
8,decoder_input,2,16,can
9,decoder_input,3,98,reset


## 4. Проста RNN-модель

У цьому блоці `nn.RNN` отримує one-hot вектори. Це напряму відповідає параметру `input_size=VOCAB_SIZE`: кожен токен представлений вектором довжини словника, де одна позиція дорівнює `1`.

Параметри моделі:

- `input_size=VOCAB_SIZE` - скільки чисел має один вхідний токен у one-hot представленні;
- `hidden_size=128` - розмір прихованого стану, тобто компактної "пам'яті" RNN;
- `num_layers=1` - один рекурентний шар;
- `nn.Linear(hidden_size, VOCAB_SIZE)` - перетворює прихований стан у logits для всіх слів словника.

`ignore_index=PAD_IDX` у loss означає: padding не впливає на якість моделі.


In [6]:
RNN_HIDDEN_SIZE = 128
RNN_NUM_LAYERS = 1

rnn_encoder = nn.RNN(
    input_size=VOCAB_SIZE,
    hidden_size=RNN_HIDDEN_SIZE,
    num_layers=RNN_NUM_LAYERS,
    batch_first=True,
).to(DEVICE)

rnn_decoder = nn.RNN(
    input_size=VOCAB_SIZE,
    hidden_size=RNN_HIDDEN_SIZE,
    num_layers=RNN_NUM_LAYERS,
    batch_first=True,
).to(DEVICE)

rnn_output_layer = nn.Linear(RNN_HIDDEN_SIZE, VOCAB_SIZE).to(DEVICE)

rnn_parameters = list(rnn_encoder.parameters()) + list(rnn_decoder.parameters()) + list(rnn_output_layer.parameters())


def rnn_forward(src_one_hot, decoder_input_one_hot):
    _, hidden = rnn_encoder(src_one_hot)
    decoder_output, _ = rnn_decoder(decoder_input_one_hot, hidden)
    return rnn_output_layer(decoder_output)


def token_accuracy(logits, targets):
    predictions = logits.argmax(dim=-1)
    mask = targets != PAD_IDX
    correct = (predictions[mask] == targets[mask]).sum().item()
    total = mask.sum().item()
    return correct / total if total else 0.0


def train_rnn_model(epochs=10, lr=0.001):
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    optimizer = torch.optim.Adam(rnn_parameters, lr=lr)
    history = []

    src_one_hot = F.one_hot(src_ids, num_classes=VOCAB_SIZE).float()
    decoder_one_hot = F.one_hot(decoder_input_ids, num_classes=VOCAB_SIZE).float()

    for epoch in range(1, epochs + 1):
        rnn_encoder.train()
        rnn_decoder.train()
        rnn_output_layer.train()

        optimizer.zero_grad()
        logits = rnn_forward(src_one_hot, decoder_one_hot)
        loss = criterion(logits.reshape(-1, VOCAB_SIZE), target_ids.reshape(-1))
        loss.backward()
        optimizer.step()

        history.append(
            {
                "epoch": epoch,
                "loss": loss.item(),
                "token_accuracy": token_accuracy(logits.detach(), target_ids),
            }
        )

    return pd.DataFrame(history)


rnn_history = train_rnn_model(epochs=10, lr=0.001)
rnn_history


,epoch,loss,token_accuracy
0,1,4.942854,0.004202
1,2,4.913589,0.016807
2,3,4.884417,0.050420
3,4,4.854014,0.063025
4,5,4.820938,0.092437
5,6,4.783380,0.142857
6,7,4.738938,0.189076
7,8,4.684486,0.172269
8,9,4.616575,0.147059
9,10,4.533299,0.176471


In [7]:
# Тест RNN-моделі на кількох запитах з навчального корпусу.
def decode_rnn_tokens(token_ids):
    words = []
    for token_id in token_ids:
        token = idx_to_token[int(token_id)]
        if token in (EOS_TOKEN, PAD_TOKEN):
            break
        if token not in SPECIAL_TOKENS:
            words.append(token)
    return " ".join(words)


def generate_rnn_test_answer(question, max_new_tokens=MAX_ANSWER_LEN):
    rnn_encoder.eval()
    rnn_decoder.eval()
    rnn_output_layer.eval()

    question_ids = encode_tokens(tokenize(question), MAX_QUESTION_LEN)
    question_tensor = torch.tensor([question_ids], dtype=torch.long, device=DEVICE)
    question_one_hot = F.one_hot(question_tensor, num_classes=VOCAB_SIZE).float()

    generated_ids = []
    current_id = torch.tensor([[SOS_IDX]], dtype=torch.long, device=DEVICE)

    with torch.no_grad():
        _, hidden = rnn_encoder(question_one_hot)

        for _ in range(max_new_tokens):
            current_one_hot = F.one_hot(current_id, num_classes=VOCAB_SIZE).float()
            decoder_output, hidden = rnn_decoder(current_one_hot, hidden)
            logits = rnn_output_layer(decoder_output[:, -1, :])
            next_id = int(logits.argmax(dim=-1).item())

            if next_id in (EOS_IDX, PAD_IDX):
                break

            generated_ids.append(next_id)
            current_id = torch.tensor([[next_id]], dtype=torch.long, device=DEVICE)

    return decode_rnn_tokens(generated_ids)


rnn_test_questions = [
    "How can I reset my password?",
    "My order is delayed",
    "How do I request a refund?",
    "How can I contact support?",
    "Is my personal data secure?",
]

expected_answers = {question: answer for question, answer in qa_pairs}

rnn_test_results = []
for question in rnn_test_questions:
    rnn_test_results.append(
        {
            "question": question,
            "expected_answer": expected_answers[question],
            "rnn_generated_answer": generate_rnn_test_answer(question),
        }
    )

pd.DataFrame(rnn_test_results)


,question,expected_answer,rnn_generated_answer
0,How can I reset my password?,You can reset your password in account settings.,you the
1,My order is delayed,Your order may take a bit longer than expected.,you the
2,How do I request a refund?,Open the order details and choose request refund.,you order
3,How can I contact support?,You can contact support through chat or email.,you the
4,Is my personal data secure?,Your personal data is protected according to o...,you the


## 5. LSTM-модель

`LSTM` має додатковий cell state, тому краще тримає інформацію на довших послідовностях, ніж проста RNN.

На відміну від RNN-блоку з one-hot, тут використовується `nn.Embedding`: замість довгого sparse-вектора модель одразу бере щільний вектор слова.

Параметри:

- `embedding_dim=64` - розмір векторного представлення одного токена;
- `hidden_size=128` - розмір прихованого стану LSTM;
- `num_layers=1` - один LSTM-шар;
- `padding_idx=PAD_IDX` - embedding для `<pad>` не навчається як змістовне слово;
- `Adam(lr=0.001)` - адаптивний оптимізатор із малим кроком навчання.


In [8]:
LSTM_EMBEDDING_DIM = 64
LSTM_HIDDEN_SIZE = 128
LSTM_NUM_LAYERS = 1

lstm_embedding = nn.Embedding(
    VOCAB_SIZE,
    LSTM_EMBEDDING_DIM,
    padding_idx=PAD_IDX,
).to(DEVICE)

lstm_encoder = nn.LSTM(
    input_size=LSTM_EMBEDDING_DIM,
    hidden_size=LSTM_HIDDEN_SIZE,
    num_layers=LSTM_NUM_LAYERS,
    batch_first=True,
).to(DEVICE)

lstm_decoder = nn.LSTM(
    input_size=LSTM_EMBEDDING_DIM,
    hidden_size=LSTM_HIDDEN_SIZE,
    num_layers=LSTM_NUM_LAYERS,
    batch_first=True,
).to(DEVICE)

lstm_output_layer = nn.Linear(LSTM_HIDDEN_SIZE, VOCAB_SIZE).to(DEVICE)

lstm_parameters = (
    list(lstm_embedding.parameters())
    + list(lstm_encoder.parameters())
    + list(lstm_decoder.parameters())
    + list(lstm_output_layer.parameters())
)


def lstm_forward(src, decoder_input):
    src_emb = lstm_embedding(src)
    decoder_emb = lstm_embedding(decoder_input)
    _, state = lstm_encoder(src_emb)
    decoder_output, _ = lstm_decoder(decoder_emb, state)
    return lstm_output_layer(decoder_output)


def train_lstm_model(epochs=15, lr=0.001):
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    optimizer = torch.optim.Adam(lstm_parameters, lr=lr)
    history = []

    for epoch in range(1, epochs + 1):
        lstm_embedding.train()
        lstm_encoder.train()
        lstm_decoder.train()
        lstm_output_layer.train()

        optimizer.zero_grad()
        logits = lstm_forward(src_ids, decoder_input_ids)
        loss = criterion(logits.reshape(-1, VOCAB_SIZE), target_ids.reshape(-1))
        loss.backward()
        optimizer.step()

        history.append(
            {
                "epoch": epoch,
                "loss": loss.item(),
                "token_accuracy": token_accuracy(logits.detach(), target_ids),
            }
        )

    return pd.DataFrame(history)


lstm_history = train_lstm_model(epochs=15, lr=0.001)
lstm_history


,epoch,loss,token_accuracy
0,1,4.945359,0.004202
1,2,4.918062,0.029412
2,3,4.890951,0.063025
3,4,4.863728,0.092437
4,5,4.836085,0.130252
5,6,4.807700,0.163866
6,7,4.778225,0.222689
7,8,4.747274,0.252101
8,9,4.714406,0.268908
9,10,4.679103,0.315126


## 6. Порівняння loss і token accuracy

`loss` показує, наскільки logits моделі далекі від правильних токенів. Менше - краще.

`token_accuracy` рахує частку правильно передбачених токенів без `<pad>`. Це базова метрика для навчального прикладу, але вона не гарантує, що вся відповідь буде логічною.


In [9]:
metrics_summary = pd.DataFrame(
    [
        {
            "model": "RNN",
            "epochs": len(rnn_history),
            "final_loss": rnn_history["loss"].iloc[-1],
            "final_token_accuracy": rnn_history["token_accuracy"].iloc[-1],
        },
        {
            "model": "LSTM",
            "epochs": len(lstm_history),
            "final_loss": lstm_history["loss"].iloc[-1],
            "final_token_accuracy": lstm_history["token_accuracy"].iloc[-1],
        },
    ]
)

metrics_summary


,model,epochs,final_loss,final_token_accuracy
0,RNN,10,4.533299,0.176471
1,LSTM,15,4.440446,0.390756


## 7. Генерація відповідей

Під час генерації decoder вже не бачить правильну відповідь. Він стартує з `<sos>`, потім кожен наступний токен бере з власного попереднього прогнозу.

`max_new_tokens` обмежує довжину відповіді, щоб модель не генерувала нескінченно. Генерація зупиняється раніше, якщо модель передбачила `<eos>`.


In [10]:
def decode_ids(ids):
    words = []
    for idx in ids:
        token = idx_to_token[int(idx)]
        if token == EOS_TOKEN:
            break
        if token not in SPECIAL_TOKENS:
            words.append(token)
    return " ".join(words)


def encode_question(question):
    ids = encode_tokens(tokenize(question), MAX_QUESTION_LEN)
    return torch.tensor([ids], dtype=torch.long, device=DEVICE)


def generate_with_rnn(question, max_new_tokens=MAX_ANSWER_LEN):
    rnn_encoder.eval()
    rnn_decoder.eval()
    rnn_output_layer.eval()

    src = encode_question(question)
    src_one_hot = F.one_hot(src, num_classes=VOCAB_SIZE).float()

    with torch.no_grad():
        _, hidden = rnn_encoder(src_one_hot)
        current_id = torch.tensor([[SOS_IDX]], dtype=torch.long, device=DEVICE)
        generated = []

        for _ in range(max_new_tokens):
            current_one_hot = F.one_hot(current_id, num_classes=VOCAB_SIZE).float()
            decoder_output, hidden = rnn_decoder(current_one_hot, hidden)
            logits = rnn_output_layer(decoder_output[:, -1, :])
            next_id = int(logits.argmax(dim=-1).item())
            if next_id in (EOS_IDX, PAD_IDX):
                break
            generated.append(next_id)
            current_id = torch.tensor([[next_id]], dtype=torch.long, device=DEVICE)

    return decode_ids(generated)


def generate_with_lstm(question, max_new_tokens=MAX_ANSWER_LEN):
    lstm_embedding.eval()
    lstm_encoder.eval()
    lstm_decoder.eval()
    lstm_output_layer.eval()

    src = encode_question(question)

    with torch.no_grad():
        _, state = lstm_encoder(lstm_embedding(src))
        current_id = torch.tensor([[SOS_IDX]], dtype=torch.long, device=DEVICE)
        generated = []

        for _ in range(max_new_tokens):
            current_emb = lstm_embedding(current_id)
            decoder_output, state = lstm_decoder(current_emb, state)
            logits = lstm_output_layer(decoder_output[:, -1, :])
            next_id = int(logits.argmax(dim=-1).item())
            if next_id in (EOS_IDX, PAD_IDX):
                break
            generated.append(next_id)
            current_id = torch.tensor([[next_id]], dtype=torch.long, device=DEVICE)

    return decode_ids(generated)


In [11]:
test_questions = [
    "How can I reset my password?",
    "My order is delayed",
    "How do I request a refund?",
    "I received the wrong item",
    "How do I unsubscribe from emails?",
]

expected_lookup = {question: answer for question, answer in qa_pairs}

generated_rows = []
for question in test_questions:
    generated_rows.append(
        {
            "question": question,
            "expected_answer": expected_lookup.get(question, ""),
            "rnn_answer": generate_with_rnn(question),
            "lstm_answer": generate_with_lstm(question),
        }
    )

generated_df = pd.DataFrame(generated_rows)
generated_df


,question,expected_answer,rnn_answer,lstm_answer
0,How can I reset my password?,You can reset your password in account settings.,you the,you can your your order the order
1,My order is delayed,Your order may take a bit longer than expected.,you the,you can your order contact support and include...
2,How do I request a refund?,Open the order details and choose request refund.,you order,you can your order the order
3,I received the wrong item,Please contact support and include your order ...,you the,you can your order contact support and include...
4,How do I unsubscribe from emails?,Use the unsubscribe link or change email prefe...,you the,you can your order the order


## 8. Таблиця спостережень

Для маленького корпусу важливо оцінювати не тільки метрику, а й текст:

- чи відповідь відповідає запиту;
- чи не повторює модель одні й ті самі слова;
- чи зупиняється генерація вчасно;
- чи не плутає різні сценарії підтримки.


In [12]:
def comment_on_answer(question, answer):
    normalized_question = normalize_text(question)
    normalized_answer = normalize_text(answer)

    if not normalized_answer:
        return "Порожня відповідь або модель одразу передбачила кінець."
    if "password" in normalized_question and "password" in normalized_answer:
        return "Є зв'язок із темою пароля."
    if "order" in normalized_question and "order" in normalized_answer:
        return "Є зв'язок із темою замовлення."
    if "refund" in normalized_question and "refund" in normalized_answer:
        return "Є зв'язок із темою повернення коштів."
    if len(normalized_answer.split()) <= 2:
        return "Відповідь занадто коротка для підтримки."
    return "Відповідь треба перевірити вручну: можлива слабка відповідність запиту."


observation_table = generated_df[["question", "lstm_answer"]].copy()
observation_table["comment"] = observation_table.apply(
    lambda row: comment_on_answer(row["question"], row["lstm_answer"]),
    axis=1,
)
observation_table


,question,lstm_answer,comment
0,How can I reset my password?,you can your your order the order,Відповідь треба перевірити вручну: можлива сла...
1,My order is delayed,you can your order contact support and include...,Є зв'язок із темою замовлення.
2,How do I request a refund?,you can your order the order,Відповідь треба перевірити вручну: можлива сла...
3,I received the wrong item,you can your order contact support and include...,Відповідь треба перевірити вручну: можлива сла...
4,How do I unsubscribe from emails?,you can your order the order,Відповідь треба перевірити вручну: можлива сла...


## 9. Transformer для порівняння

`distilgpt2` - переднавчена Transformer-модель для продовження тексту. Вона не fine-tuned саме на службі підтримки, тому її відповідь може бути граматично зв'язною, але не завжди бізнес-коректною.

Ключові параметри генерації:

- `max_new_tokens` - скільки нових токенів дозволено згенерувати;
- `do_sample=True` - модель вибирає не завжди найімовірніший токен, тому відповіді різноманітніші;
- `temperature` - керує випадковістю: менше значення робить текст стабільнішим;
- `top_p` - nucleus sampling, модель вибирає з найімовірнішої частини розподілу.

Якщо `transformers` не встановлено або модель не завантажена, клітинка не зупиняє notebook, а показує причину.


In [13]:
from transformers import pipeline, set_seed

set_seed(SEED)
generator = pipeline("text-generation", model="distilgpt2")
rows = []

for question in test_questions:
    prompt = f"Customer: {question}\nSupport:"
    start = time.perf_counter()
    output = generator(
        prompt,
        max_new_tokens=35,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=generator.tokenizer.eos_token_id,
    )[0]["generated_text"]
    elapsed = time.perf_counter() - start
    answer = output.split("Support:", 1)[-1].strip()
    rows.append(
        {
            "question": question,
            "transformer_answer": answer,
            "seconds": round(elapsed, 2),
        }
    )

transformer_df = pd.DataFrame(rows)
transformer_df


/Users/dmitrostefan/Edu/PythonAI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 76/76 [00:00<00:00, 8899.14it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_p', 'temperature', 'pad_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=35) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2

,question,transformer_answer,seconds
0,How can I reset my password?,You can reset your password by changing your p...,0.61
1,My order is delayed,My order is delayed\n\nThank you for your supp...,0.53
2,How do I request a refund?,I am currently working on a refund.\nHow do I ...,0.42
3,I received the wrong item,The seller of the item was very happy with the...,0.47
4,How do I unsubscribe from emails?,If you have any questions or comments please f...,0.31


## 10. Порівняння підходів

RNN і LSTM тут навчаються з нуля на 25 прикладах, тому вони бачать тільки вузький навчальний світ. Transformer уже має мовні знання з великого корпусу, але без fine-tuning не знає правил конкретної компанії.


In [14]:
comparison = pd.DataFrame(
    [
        {
            "criterion": "Якість відповідей",
            "RNN": "Слабка на довших залежностях, швидко повторює часті токени.",
            "LSTM": "Краще тримає послідовність, але 25 прикладів замало для стабільної генерації.",
            "Transformer": "Зазвичай зв'язніший текст, але без fine-tuning може вигадувати деталі.",
        },
        {
            "criterion": "Збереження контексту",
            "RNN": "Обмежене прихованим станом, легко втрачає ранні слова.",
            "LSTM": "Краще за RNN завдяки gates і cell state.",
            "Transformer": "Attention дозволяє напряму враховувати різні частини prompt.",
        },
        {
            "criterion": "Час навчання",
            "RNN": "Швидке на маленькому корпусі.",
            "LSTM": "Трохи повільніше за RNN.",
            "Transformer": "Переднавчення дуже дороге; inference готової моделі простіший.",
        },
        {
            "criterion": "Що покращити",
            "RNN": "Збільшити корпус або перейти до LSTM/attention.",
            "LSTM": "Додати більше діалогів, validation split, beam search або attention.",
            "Transformer": "Fine-tuning або RAG з базою знань підтримки.",
        },
    ]
)

comparison


,criterion,RNN,LSTM,Transformer
0,Якість відповідей,"Слабка на довших залежностях, швидко повторює ...","Краще тримає послідовність, але 25 прикладів з...","Зазвичай зв'язніший текст, але без fine-tuning..."
1,Збереження контексту,"Обмежене прихованим станом, легко втрачає ранн...",Краще за RNN завдяки gates і cell state.,Attention дозволяє напряму враховувати різні ч...
2,Час навчання,Швидке на маленькому корпусі.,Трохи повільніше за RNN.,Переднавчення дуже дороге; inference готової м...
3,Що покращити,Збільшити корпус або перейти до LSTM/attention.,"Додати більше діалогів, validation split, beam...",Fine-tuning або RAG з базою знань підтримки.


## 11. Збереження LSTM-моделі

Зберігаємо не тільки ваги, а й словник та основні розміри. Без словника модель не знатиме, який індекс відповідає якому слову.


In [15]:
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODEL_DIR / "supportflow_lstm.pth"

torch.save(
    {
        "embedding_state_dict": lstm_embedding.state_dict(),
        "encoder_state_dict": lstm_encoder.state_dict(),
        "decoder_state_dict": lstm_decoder.state_dict(),
        "output_layer_state_dict": lstm_output_layer.state_dict(),
        "token_to_idx": token_to_idx,
        "idx_to_token": idx_to_token,
        "max_question_len": MAX_QUESTION_LEN,
        "max_answer_len": MAX_ANSWER_LEN,
        "vocab_size": VOCAB_SIZE,
        "embedding_dim": LSTM_EMBEDDING_DIM,
        "hidden_size": LSTM_HIDDEN_SIZE,
        "num_layers": LSTM_NUM_LAYERS,
    },
    MODEL_PATH,
)

print(f"Saved to {MODEL_PATH}")


Saved to models/supportflow_lstm.pth


## Висновок

Для цього корпусу `LSTM` є кращою навчальною baseline-моделлю, ніж проста `RNN`, бо краще зберігає стан послідовності. Для реального SupportFlow AI найпрактичніший шлях - Transformer-підхід із fine-tuning або RAG, але RNN/LSTM корисні для розуміння того, як текст перетворюється на послідовність прогнозів.
